1. 도구 준비하기 (Import)

In [ ]:
import cv2  # OpenCV: 이미지/비디오를 다루는 도구 (눈 역할)
import mediapipe as mp # MediaPipe: 구글이 만든 AI 도구 (뇌 역할)

# AI 모델 중 '얼굴 검출' 기능을 꺼내옵니다.
mp_face_detection = mp.solutions.face_detection
# 검출된 결과(박스, 점)를 화면에 쉽게 그리게 도와주는 도구입니다.
mp_drawing = mp.solutions.drawing_utils

2. 비디오 파일 불러오기

In [ ]:
# 'treeman_talk.mp4' 영상을 읽어오라는 명령입니다. 
# 만약 실시간 웹캠을 쓰려면 숫지 0을 넣으면 됩니다.
cap = cv2.VideoCapture('treeman_talk.mp4')

3. AI 모델 초기화 (설정)

In [ ]:
# FaceDetection 기능을 실행합니다.
# model_selection=0: 가까운 거리(2m 이내)의 얼굴에 최적화
# min_detection_confidence=0.5: 50% 이상 확신이 들 때만 얼굴이라고 판단해라!
with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5) as face_detection:

4. 반복문 (영상은 사진의 연속이니까요!)

In [ ]:
while cap.isOpened(): # 비디오가 정상적으로 열려 있는 동안 계속 반복해라!
    success, image = cap.read() # 비디오에서 사진 한 장(Frame)을 가져옵니다.
    if not success: # 사진을 못 가져왔다면 (영상이 끝났다면)
      break # 반복문을 탈출해라!

5. AI를 위한 이미지 전처리 (가장 중요한 부분!)

In [ ]:
# 효율성을 위해 이미지를 '읽기 전용'으로 잠시 바꿉니다. (처리 속도 향상)
    image.flags.writeable = False
    
    # OpenCV는 색상을 BGR(파랑, 초록, 빨강) 순서로 읽지만, 
    # MediaPipe(AI)는 RGB 순서를 원합니다. 그래서 색상 순서를 바꿔줍니다.
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 드디어 AI에게 "이 사진에서 얼굴 좀 찾아줘!"라고 시킵니다.
    results = face_detection.process(image)

6. 결과 화면에 그리기

In [ ]:
# 다시 이미지에 그림을 그려야 하니 '쓰기 가능'으로 바꿉니다.
    image.flags.writeable = True
    
    # AI 처리가 끝났으니 우리 눈에 익숙한 BGR 색상으로 다시 되돌립니다.
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    # 만약 AI가 얼굴을 찾았다면(results.detections가 있다면)?
    if results.detections:
      for detection in results.detections: # 찾은 얼굴 개수만큼 반복하면서
        # 미리 준비된 도구로 얼굴에 사각형 박스와 6개의 점을 그려줍니다.
        mp_drawing.draw_detection(image, detection)

7. 화면 보여주기 및 종료

In [ ]:
# 'MediaPipe Face Detection'이라는 이름의 창에 결과 이미지를 보여줍니다.
    cv2.imshow('MediaPipe Face Detection', image)
    
    # 키보드의 'q'를 누르면 창을 닫고 종료합니다. (1ms 동안 입력을 기다림)
    if cv2.waitKey(1) & 0xFF == ord('q'):
      break

8. 뒷정리 (자원 반납)

In [ ]:
cap.release() # 비디오 읽기 기능을 종료하고 메모리를 해제합니다.
cv2.destroyAllWindows() # 열려 있는 모든 창을 닫습니다.

In [84]:
import cv2
import mediapipe as mp
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# For webcam input:
cap = cv2.VideoCapture('treeman_talk.mp4')
with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5) as face_detection:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      break

    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(image)

    # Draw the face detection annotations on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if results.detections:
      for detection in results.detections:
        mp_drawing.draw_detection(image, detection)
    # Flip the image horizontally for a selfie-view display.
    cv2.imshow('MediaPipe Face Detection', image)
    if cv2.waitKey(1) & 0xFF == ord('q'):
      break
cap.release()
cv2.destroyAllWindows()

In [83]:
import cv2
import mediapipe as mp

mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture('treeman_talk.mp4')

with mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5) as face_detection:
    
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.detections:
            for detection in results.detections:
                detection = results.detections[0]
                # 1. 전체 박스 및 기본 랜드마크 그리기 (기존 코드)
                # mp_drawing.draw_detection(image, detection)

                # 2. 특정 위치값(눈, 입) 직접 추출하기
                keypoints = detection.location_data.relative_keypoints
                
                # 이미지의 실제 가로, 세로 크기 확인
                h, w, c = image.shape

                # 오른쪽 눈 (인덱스 0), 왼쪽 눈 (인덱스 1), 입 중심 (인덱스 3)
                target_indices = [0, 1, 3] 
                
                for idx in target_indices:
                    keypoint = keypoints[idx]
                    
                    # 상대 좌표(0.0 ~ 1.0)를 실제 픽셀 좌표로 변환
                    cx, cy = int(keypoint.x * w), int(keypoint.y * h)
                    
                    # 3. 원 그리기 (이미지, 중심좌표, 반지름, 색상(BGR), 두께)
                    # 눈은 파란색, 입은 빨간색 등으로 구분하고 싶다면 아래처럼 분기 가능
                    if idx == 3: # 입
                        cv2.circle(image, (cx, cy), 10, (0, 0, 255), -1) # 빨간색 채워진 원
                    else: # 눈
                        cv2.circle(image, (cx, cy), 8, (255, 0, 0), -1) # 파란색 채워진 원
                    
                    # (선택사항) 좌표값 텍스트 출력
                    # cv2.putText(image, f'({cx}, {cy})', (cx, cy - 10), 
                    #             cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        cv2.imshow('MediaPipe Face Keypoints', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

얼굴의 특정 부위(눈, 코, 입 등)의 위치값을 가져오려면 MediaPipe Face Detection이 제공하는 RelativeKeypoint 정보를 활용해야 합니다.

현재 사용 중인 FaceDetection 모델은 얼굴 전체 박스뿐만 아니라, 주요 6개 지점(오른쪽 눈, 왼쪽 눈, 코 끝, 입 중심, 오른쪽 귀, 왼쪽 귀)의 좌표를 제공합니다.

1. 주요 포인트의 인덱스 번호
MediaPipe에서 정의한 6개 핵심 포인트(Keypoints)의 번호는 다음과 같습니다:

0: 오른쪽 눈 (Right Eye)

1: 왼쪽 눈 (Left Eye)

2: 코 끝 (Nose Tip)

3: 입 중심 (Mouth Center)

4: 오른쪽 귀 (Right Ear Tragion)

5: 왼쪽 귀 (Left Ear Tragion)

2. 위치값 추출 및 원 그리기 코드
results.detections 내부의 location_data.relative_keypoints를 순회하며 좌표를 계산하고 cv2.circle로 화면에 그려주는 로직을 추가한 코드입니다.

💡 핵심 설명
1. 좌표 변환: MediaPipe는 좌표를 0에서 1 사이의 비율값(relative_keypoints)으로 줍니다. 그래서 반드시 이미지의 실제 가로(w)와 세로(h) 길이를 곱해줘야 우리가 아는 픽셀 좌표가 나옵니다.
- $x\_pixel = x\_relative \times width$
- $y\_pixel = y\_relative \times height$
2. cv2.circle 함수:
- image: 그릴 대상
- (cx, cy): 원의 중심점
- 10: 반지름(Radius)
- (0, 0, 255): BGR 색상 (이 경우 빨간색)
- -1: 두께(Thickness). -1을 주면 내부를 꽉 채웁니다. 양수를 주면 테두리만 그립니다.

만약 눈과 입의 **윤곽선 전체(수십 개의 점)**를 더 정밀하게 따고 싶으시다면, FaceDetection 대신 FaceMesh 모델을 사용해야 합니다. 6개의 점으로 충분하신가요, 아니면 더 세밀한 그물망 좌표가 필요하신가요?


In [23]:
import cv2
import mediapipe as mp
import numpy as np

# 1. 풍선껌 이미지 로드 (반드시 투명 배경 PNG여야 함)
# cv2.IMREAD_UNCHANGED 옵션이 있어야 알파 채널을 가져옵니다.
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)

mp_face_detection = mp.solutions.face_detection
cap = cv2.VideoCapture('treeman_talk.mp4')

# --- 조절 가능한 변수들 ---
bubble_scale = 0.2  # 풍선껌 크기 배율
offset_y = -120       # 입 아래로 내리려면 양수, 위로 올리려면 음수
offset_x = 150        # 좌우 위치 조절
# -----------------------

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.8) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, c = image.shape
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_detection.process(image_rgb)

        if results.detections:
            for detection in results.detections:
                keypoints = detection.location_data.relative_keypoints
                
                # 입 중심점 (인덱스 3)
                mouth_kp = keypoints[3]
                cx, cy = int(mouth_kp.x * w) + offset_x, int(mouth_kp.y * h) + offset_y

                # --- 합성 프로세스 ---
                # 1. 풍선껌 크기 조절
                bw = int(w * bubble_scale)
                bh = int(bw * (bubble_img.shape[0] / bubble_img.shape[1]))
                bubble_resized = cv2.resize(bubble_img, (bw, bh))

                # 2. 합성할 영역 계산 (중심점 기준)
                y1, y2 = cy - bh//2, cy + bh//2
                x1, x2 = cx - bw//2, cx + bw//2

                # 화면 밖으로 나가는 경우 예외 처리
                if y1 < 0 or y2 > h or x1 < 0 or x2 > w:
                    continue

                # 3. 알파 채널을 이용한 오버레이
                alpha_s = bubble_resized[:, :, 3] / 255.0  # 풍선껌의 투명도
                alpha_l = 1.0 - alpha_s                   # 배경의 투명도 (반전)

                for c in range(0, 3): # B, G, R 채널 각각 합성
                    image[y1:y2, x1:x2, c] = (alpha_s * bubble_resized[:, :, c] +
                                              alpha_l * image[y1:y2, x1:x2, c])

        cv2.imshow('Bubble Gum Overlay', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [14]:
import cv2
import mediapipe as mp
import numpy as np

# 이미지 로드 (파일명은 본인의 파일에 맞게 수정하세요)
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)
eye_img = cv2.imread('heart.png', cv2.IMREAD_UNCHANGED)

mp_face_detection = mp.solutions.face_detection
cap = cv2.VideoCapture('treeman_talk.mp4')

# --- 조절 가능한 변수들 ---
eye_scale = 0.05    # 눈 이미지 크기
mouth_scale = 0.2  # 입 풍선껌 크기
offset_y = -120     # 입 위치 미세조정
offset_x = 170        # 좌우 위치 조절
# alpha = 0.6
# -----------------------

def overlay_image(background, overlay, cx, cy, scale):
    h, w, _ = background.shape
    # print(background.shape)
    # 크기 조절
    bw = int(w * scale)
    bh = int(bw * (overlay.shape[0] / overlay.shape[1]))
    img_resized = cv2.resize(overlay, (bw, bh))

    # 합성 영역 계산
    y1, y2 = cy - bh//2, cy + bh//2
    x1, x2 = cx - bw//2, cx + bw//2

    # 범위 체크 및 합성
    if 0 <= y1 and y2 <= h and 0 <= x1 and x2 <= w:
        alpha_s = img_resized[:, :, 3] / 255.0
        alpha_l = 1.0 - alpha_s
        for c in range(3):
            background[y1:y2, x1:x2, c] = (alpha_s * img_resized[:, :, c] + 
                                           alpha_l * background[y1:y2, x1:x2, c])
    return background

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.8) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, _ = image.shape
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.detections:
            # 가장 확실한 첫 번째 사람만 타겟팅
            detection = results.detections[0]
            keypoints = detection.location_data.relative_keypoints

            # 1. 양쪽 눈에 이미지 넣기 (인덱스 0: 우측, 1: 좌측)
            for i in [0, 1]:
                ex, ey = int(keypoints[i].x * w), int(keypoints[i].y * h)
                image = overlay_image(image, eye_img, ex, ey, eye_scale)

            # 2. 입에 이미지 넣기 (인덱스 3)
            mx, my = int(keypoints[3].x * w) + offset_x, int(keypoints[3].y * h) + offset_y
            image = overlay_image(image, bubble_img, mx, my, mouth_scale)

        cv2.imshow('MediaPipe Face Overlay', image)
        if cv2.waitKey(30) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [26]:
import cv2
import mediapipe as mp
import numpy as np

# 이미지 로드 (파일명은 본인의 파일에 맞게 수정하세요)
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)
eye_img = cv2.imread('heart2.png', cv2.IMREAD_UNCHANGED)

mp_face_detection = mp.solutions.face_detection
cap = cv2.VideoCapture('treeman_talk.mp4')

# --- 조절 가능한 변수들 ---
eye_scale = 0.05    # 눈 이미지 크기
mouth_scale = 0.2  # 입 풍선껌 크기
offset_y = -120     # 입 위치 미세조정
offset_x = 170        # 좌우 위치 조절
# -----------------------

def overlay_image(background, overlay, cx, cy, scale, global_alpha=0.99):
    h, w, _ = background.shape
    
    # 1. 크기 조절
    bw = int(w * scale)
    bh = int(bw * (overlay.shape[0] / overlay.shape[1]))
    if bw <= 0 or bh <= 0: return background
    img_resized = cv2.resize(overlay, (bw, bh))

    # 2. 가우시안 블러 적용 (경계를 부드럽게)
    # 이미지 크기에 따라 커널 사이즈를 조절 (항상 홀수여야 함)
    kernel_size = (int(bw * 0.02) // 2 * 2 + 1) 
    if kernel_size > 1:
        img_resized = cv2.GaussianBlur(img_resized, (kernel_size, kernel_size), 0)
    
    # 아주 미세하게 경계만 다듬고 싶을 때 (3x3 또는 5x5 추천)
    # kernel_size = 3 
    # img_resized = cv2.GaussianBlur(img_resized, (kernel_size, kernel_size), 0)

    # 합성 영역 계산?
    y1, y2 = cy - bh//2, cy + bh//2
    x1, x2 = cx - bw//2, cx + bw//2

    # 범위 체크 (이미지가 화면 밖으로 나가는 것 방지)
    if x1 < 0 or y1 < 0 or x2 > w or y2 > h:
        return background

    # 3. 알파 채널 분리 및 블렌딩
    # PNG의 자체 투명도(alpha_s)
    alpha_s = img_resized[:, :, 3] / 255.0
    
    # 사용자가 지정한 전체 투명도(global_alpha) 적용
    combined_alpha = alpha_s * global_alpha
    inv_alpha = 1.0 - combined_alpha

    # ROI 추출 후 합성
    for c in range(3):
        background[y1:y2, x1:x2, c] = (combined_alpha * img_resized[:, :, c] + 
                                       inv_alpha * background[y1:y2, x1:x2, c])
    
    return background

# --- [추가] 스무딩을 위한 좌표 저장 변수 초기화 ---
# 눈(좌, 우)과 입의 좌표를 저장할 딕셔너리
prev_coords = {
    'eye_0': None, # 우측 눈
    'eye_1': None, # 좌측 눈
    'mouth': None  # 입
}
# 스무딩 계수 (0.0 ~ 1.0) 
# 0.1에 가까울수록 매우 부드럽지만 반응이 느림 (잔상 남음)
# 0.7~0.8 정도가 적당합니다.
SMOOTHING = 0.5 

def get_smoothed_coords(key, curr_x, curr_y):
    if prev_coords[key] is None:
        prev_coords[key] = (curr_x, curr_y)
        return curr_x, curr_y
    
    prev_x, prev_y = prev_coords[key]
    # 공식: (이전값 * 가중치) + (현재값 * (1 - 가중치))
    smooth_x = int(prev_x * SMOOTHING + curr_x * (1 - SMOOTHING))
    smooth_y = int(prev_y * SMOOTHING + curr_y * (1 - SMOOTHING))
    
    prev_coords[key] = (smooth_x, smooth_y) # 업데이트
    return smooth_x, smooth_y

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.8) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, _ = image.shape
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.detections:
            detection = results.detections[0]
            keypoints = detection.location_data.relative_keypoints

            # 1. 양쪽 눈 스무딩 및 합성
            for i in [0, 1]:
                raw_ex, raw_ey = int(keypoints[i].x * w), int(keypoints[i].y * h)
                # 스무딩 함수 호출
                ex, ey = get_smoothed_coords(f'eye_{i}', raw_ex, raw_ey)
                image = overlay_image(image, eye_img, ex, ey, eye_scale)

            # 2. 입 스무딩 및 합성
            raw_mx = int(keypoints[3].x * w) + offset_x
            raw_my = int(keypoints[3].y * h) + offset_y
            mx, my = get_smoothed_coords('mouth', raw_mx, raw_my)
            image = overlay_image(image, bubble_img, mx, my, mouth_scale)

        cv2.imshow('MediaPipe Face Overlay', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

3. 특정 영역(알파 채널)만 블러 처리하기
현재는 이미지 전체를 블러 처리하고 있습니다. 만약 "이미지 안쪽은 선명하고 외곽선만 자연스럽게" 하고 싶다면, 알파 채널에만 블러를 적용하는 것이 기술적인 정석입니다.

def overlay_image(background, overlay, cx, cy, scale, global_alpha=0.8):
    # ... (크기 조절 로직 동일) ...
    
    img_resized = cv2.resize(overlay, (bw, bh))
    
    # [수정] 이미지 전체가 아닌 '투명도(Alpha)'만 블러 처리
    alpha_channel = img_resized[:, :, 3]
    # 외곽선을 부드럽게 만들기 위해 알파 채널에만 3x3 블러 적용
    alpha_channel = cv2.GaussianBlur(alpha_channel, (3, 3), 0)
    
    # ... (합성 영역 계산 동일) ...

    # 블러 처리된 알파 채널 사용
    alpha_s = alpha_channel / 255.0
    combined_alpha = alpha_s * global_alpha
    inv_alpha = 1.0 - combined_alpha

    for c in range(3):
        background[y1:y2, x1:x2, c] = (combined_alpha * img_resized[:, :, c] + 
                                       inv_alpha * background[y1:y2, x1:x2, c])
    return background

In [25]:
import cv2
import mediapipe as mp
import numpy as np
import math

# 1. 이미지 로드
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)
mask_img = cv2.imread('mask.png', cv2.IMREAD_UNCHANGED)

mp_face_detection = mp.solutions.face_detection
cap = cv2.VideoCapture('treeman_talk.mp4')

# 조절 변수
mask_scale = 2.5      # 가면 크기 (눈 사이 거리의 배수)
mask_offset_y = -30   # 가면 상하 위치 (마이너스는 위로, 플러스는 아래로)
mask_offset_x = -10
mouth_scale = 0.2
offset_y = -100  # 입 위치 조정 (영상에 맞춰 수정)
offset_x = 200
SMOOTHING = 0.5 

# 기존 입 합성을 위한 함수 (꼭 필요합니다)
def overlay_image(background, overlay, cx, cy, scale):
    if overlay is None: return background
    h, w, _ = background.shape
    bw = int(w * scale)
    bh = int(bw * (overlay.shape[0] / overlay.shape[1]))
    img_resized = cv2.resize(overlay, (bw, bh))

    y1, y2 = cy - bh//2, cy + bh//2
    x1, x2 = cx - bw//2, cx + bw//2

    # 범위 보정 (화면 밖으로 나가도 잘리지 않게 처리)
    if x1 < 0 or y1 < 0 or x2 > w or y2 > h: return background

    alpha_s = img_resized[:, :, 3] / 255.0
    inv_alpha = 1.0 - alpha_s
    for c in range(3):
        background[y1:y2, x1:x2, c] = (alpha_s * img_resized[:, :, c] + inv_alpha * background[y1:y2, x1:x2, c])
    return background

# 가면에 특화된 함수 (회전/크기 조절)
def overlay_mask(background, mask, p1, p2, scale):
    if mask is None: return background
    h, w, _ = background.shape
    
    # 1. 중심점 계산 및 오프셋 적용
    cx = (p1[0] + p2[0]) // 2 + mask_offset_x
    cy = (p1[1] + p2[1]) // 2 + mask_offset_y  # <--- 여기서 위치 조절!
    
    dist = math.hypot(p2[0] - p1[0], p2[1] - p1[1])    
    mask_width = int(dist * scale) # 가면 크기 배율
    if mask_width <= 0: return background
    
    # 두 눈의 기울기 (p1: 우측눈, p2: 좌측눈 기준)
    angle = math.degrees(math.atan2(p2[1] - p1[1], p2[0] - p1[0]))
    
    # 2. 이미지 회전 및 크기 조절
    h_m, w_m = mask.shape[:2]
    mask_height = int(mask_width * (h_m / w_m))
    resized_mask = cv2.resize(mask, (mask_width, mask_height))
    
    M = cv2.getRotationMatrix2D((mask_width // 2, mask_height // 2), -angle, 1.0)
    rotated_mask = cv2.warpAffine(resized_mask, M, (mask_width, mask_height), 
                                  flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))

    # 3. 합성 영역 계산
    # [수정된 부분] 슬라이싱 크기를 rotated_mask의 실제 크기로 고정합니다.
    rh, rw = rotated_mask.shape[:2]
    y1, y2 = cy - rh // 2, (cy - rh // 2) + rh
    x1, x2 = cx - rw // 2, (cx - rw // 2) + rw
    
    # 화면 밖 예외 처리
    if x1 < 0 or y1 < 0 or x2 > w or y2 > h: return background

    # 4. 알파 블렌딩
    alpha_s = rotated_mask[:, :, 3] / 255.0
    inv_alpha = 1.0 - alpha_s
    
    # 배경의 관심 영역(ROI) 추출
    roi = background[y1:y2, x1:x2]
    
    for c in range(3):
        # 이제 두 영역의 크기가 무조건 일치합니다.
        roi[:, :, c] = (alpha_s * rotated_mask[:, :, c] + inv_alpha * roi[:, :, c])
    
    background[y1:y2, x1:x2] = roi
    return background

# 스무딩 관련
prev_coords = {'eye_0': None, 'eye_1': None, 'mouth': None}

def get_smoothed_coords(key, curr_x, curr_y):
    if prev_coords[key] is None:
        prev_coords[key] = (curr_x, curr_y)
        return curr_x, curr_y
    prev_x, prev_y = prev_coords[key]
    smooth_x = int(prev_x * SMOOTHING + curr_x * (1 - SMOOTHING))
    smooth_y = int(prev_y * SMOOTHING + curr_y * (1 - SMOOTHING))
    prev_coords[key] = (smooth_x, smooth_y)
    return smooth_x, smooth_y

# 메인 루프
with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.8) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, _ = image.shape
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.detections:
            detection = results.detections[0]
            keypoints = detection.location_data.relative_keypoints

            # 오른쪽 눈(0), 왼쪽 눈(1) 좌표 스무딩
            p0 = get_smoothed_coords('eye_0', int(keypoints[0].x * w), int(keypoints[0].y * h))
            p1 = get_smoothed_coords('eye_1', int(keypoints[1].x * w), int(keypoints[1].y * h))

            # 1. 가면 합성
            image = overlay_mask(image, mask_img, p0, p1, mask_scale)

            # 2. 입 합성
            raw_mx = int(keypoints[3].x * w) + offset_x
            raw_my = int(keypoints[3].y * h) + offset_y
            mx, my = get_smoothed_coords('mouth', raw_mx, raw_my)
            image = overlay_image(image, bubble_img, mx, my, mouth_scale)

        cv2.imshow('MediaPipe Face Overlay', image)
        if cv2.waitKey(100) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [56]:
import cv2
import mediapipe as mp
import numpy as np
import math

# 1. 이미지 로드
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)
mask_img = cv2.imread('mask.png', cv2.IMREAD_UNCHANGED)

mp_face_detection = mp.solutions.face_detection
cap = cv2.VideoCapture('treeman_talk.mp4')

# --- [설정 변수: 여기서 수치를 조절하세요] ---
FIXED_MASK_WIDTH = 280  # 가면 가로 크기 고정 (부르르 떨리는 현상 방지)
mask_offset_y = -65     # 가면 위/아래 위치 조절
mask_offset_x = 0       # 가면 좌/우 위치 조절

mouth_scale = 0.2       # 입 이미지 크기
offset_y = -100         # 입 위/아래 위치
offset_x = 200          # 입 좌/우 위치

SMOOTHING = 0.5         # 좌표 안정화 (0.5가 적당)

MASK_BLUR_SIZE = 1      # 블러 강도 (3, 5, 7 등 홀수만 가능. 0이나 1이면 블러 없음)
MASK_ALPHA = 0.9        # 전체 투명도 (1.0이면 완전 불투명, 낮을수록 원본과 섞임)
# ------------------------------------------

# 이전 좌표 저장용 (스무딩용)
prev_coords = {'eye_0': None, 'eye_1': None, 'mouth': None}

def get_smoothed_coords(key, curr_x, curr_y):
    if prev_coords[key] is None:
        prev_coords[key] = (curr_x, curr_y)
        return curr_x, curr_y
    px, py = prev_coords[key]
    sx = int(px * SMOOTHING + curr_x * (1 - SMOOTHING))
    sy = int(py * SMOOTHING + curr_y * (1 - SMOOTHING))
    prev_coords[key] = (sx, sy)
    return sx, sy

def overlay_mask_refined(background, mask, p0, p1):
    if mask is None: return background
    bg_h, bg_w = background.shape[:2]
    
    # 1. 소수점 좌표 그대로 사용 (float 유지)
    cx = (p0[0] + p1[0]) / 2.0 + mask_offset_x
    cy = (p0[1] + p1[1]) / 2.0 + mask_offset_y
    angle = math.degrees(math.atan2(p1[1] - p0[1], p1[0] - p0[0]))
    
    # 2. 리사이즈
    h_m, w_m = mask.shape[:2]
    mask_width = FIXED_MASK_WIDTH
    mask_height = int(mask_width * (h_m / w_m))
    resized = cv2.resize(mask, (mask_width, mask_height))

    # 3. [블러 조절] 외곽선 희멀떡 방지
    if MASK_BLUR_SIZE > 1:
        # 이미지 전체가 아닌 알파 채널(가장자리)에만 약하게 블러 적용
        alpha_channel = resized[:, :, 3]
        alpha_channel = cv2.GaussianBlur(alpha_channel, (MASK_BLUR_SIZE, MASK_BLUR_SIZE), 0)
        resized[:, :, 3] = alpha_channel

    # 4. 회전 행렬 (소수점 cx, cy 기준 중심으로 이동)
    # 픽셀 밀림을 방지하기 위해 WarpAffine에서 소수점 이동(Translation)을 포함시킵니다.
    M = cv2.getRotationMatrix2D((mask_width / 2, mask_height / 2), -angle, 1.0)
    
    # 실제 배경에서의 위치로 이동시키기 위한 행렬 수정
    M[0, 2] += (cx - mask_width / 2)
    M[1, 2] += (cy - mask_height / 2)

    # 5. 소수점 단위로 부드럽게 변환 (Sub-pixel)
    # 이 과정에서 int 변환 없이 부드럽게 배치됩니다.
    rotated = cv2.warpAffine(resized, M, (bg_w, bg_h), 
                              flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))

    # 6. 최종 합성 (알파 블렌딩 조절)
    alpha_mask = (rotated[:, :, 3] / 255.0) * MASK_ALPHA # 여기서 블렌딩 강도 조절
    inv_alpha = 1.0 - alpha_mask
    
    for c in range(3):
        background[:, :, c] = (alpha_mask * rotated[:, :, c] + inv_alpha * background[:, :, c]).astype(np.uint8)
    
    return background

# 입 전용 합성 함수
def overlay_image(background, overlay, cx, cy, scale):
    if overlay is None: return background
    h, w, _ = background.shape
    bw = int(w * scale)
    bh = int(bw * (overlay.shape[0] / overlay.shape[1]))
    img_resized = cv2.resize(overlay, (bw, bh))

    y1, y2 = cy - bh//2, cy + bh//2
    x1, x2 = cx - bw//2, cx + bw//2

    if x1 < 0 or y1 < 0 or x2 > w or y2 > h: return background

    alpha_s = img_resized[:, :, 3] / 255.0
    inv_alpha = 1.0 - alpha_s
    for c in range(3):
        background[y1:y2, x1:x2, c] = (alpha_s * img_resized[:, :, c] + inv_alpha * background[y1:y2, x1:x2, c])
    return background

# 메인 루프
with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, _ = image.shape
        results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.detections:
            detection = results.detections[0]
            keypoints = detection.location_data.relative_keypoints

            # 1. 눈 좌표 스무딩 (index 0: 우측, 1: 좌측)
            p0 = get_smoothed_coords('eye_0', int(keypoints[0].x * w), int(keypoints[0].y * h))
            p1 = get_smoothed_coords('eye_1', int(keypoints[1].x * w), int(keypoints[1].y * h))

            # 2. 가면 합성 (안전한 고정 방식 함수 호출)
            image = overlay_mask_fixed(image, mask_img, p0, p1)

            # 3. 입 좌표 스무딩 및 합성
            raw_mx = int(keypoints[3].x * w) + offset_x
            raw_my = int(keypoints[3].y * h) + offset_y
            mx, my = get_smoothed_coords('mouth', raw_mx, raw_my)
            image = overlay_image(image, bubble_img, mx, my, mouth_scale)

        cv2.imshow('MediaPipe Face Overlay', image)
        if cv2.waitKey(30) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [55]:
mask_img.shape

(345, 208, 4)

In [80]:
import cv2
import mediapipe as mp
import numpy as np
import math

# 1. 이미지 로드 및 "사전 전처리" (루프 밖에서 한 번만!)
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)
raw_mask = cv2.imread('mask.png', cv2.IMREAD_UNCHANGED)

# [최적화] 루프에 들어가기 전 테두리 미리 깎아두기
if raw_mask is not None and raw_mask.shape[2] == 4:
    kernel = np.ones((2,2), np.uint8)
    channels = list(cv2.split(raw_mask))
    channels[3] = cv2.erode(channels[3], kernel, iterations=1) # 미리 깎기
    pre_processed_mask = cv2.merge(channels)
    # 미리 블러까지 적용 (원한다면)
    pre_processed_mask = cv2.GaussianBlur(pre_processed_mask, (3, 3), 0)
else:
    pre_processed_mask = raw_mask

mp_face_mesh = mp.solutions.face_mesh
cap = cv2.VideoCapture('treeman_talk.mp4')

# --- [설정 변수] ---
FIXED_MASK_WIDTH = 270
mask_offset_y = -60
mask_offset_x = 0
mouth_scale = 0.2
offset_y = -100
offset_x = 220
SMOOTHING = 0.5 
MASK_ALPHA = 0.93
# ------------------

prev_coords = {'eye_0': None, 'eye_1': None, 'mouth': None}

def get_smoothed_coords(key, curr_x, curr_y):
    if prev_coords[key] is None:
        prev_coords[key] = (curr_x, curr_y)
        return curr_x, curr_y
    px, py = prev_coords[key]
    sx = px * SMOOTHING + curr_x * (1 - SMOOTHING)
    sy = py * SMOOTHING + curr_y * (1 - SMOOTHING)
    prev_coords[key] = (sx, sy)
    return sx, sy

# 최적화된 합성 함수 (전처리 로직 제거)
def overlay_mask_fast(background, mask, p0, p1):
    if mask is None: return background
    bg_h, bg_w = background.shape[:2]
    
    cx = (p0[0] + p1[0]) / 2.0 + mask_offset_x
    cy = (p0[1] + p1[1]) / 2.0 + mask_offset_y
    angle = math.degrees(math.atan2(p1[1] - p0[1], p1[0] - p0[0]))
    
    h_m, w_m = mask.shape[:2]
    mask_width = FIXED_MASK_WIDTH
    mask_height = int(mask_width * (h_m / w_m))
    
    # [최적화] 보간법을 Linear로 변경 (속도 향상)
    resized = cv2.resize(mask, (mask_width, mask_height), interpolation=cv2.INTER_LINEAR)

    M = cv2.getRotationMatrix2D((mask_width / 2, mask_height / 2), -angle, 1.0)
    M[0, 2] += (cx - mask_width / 2)
    M[1, 2] += (cy - mask_height / 2)

    rotated = cv2.warpAffine(resized, M, (bg_w, bg_h), 
                              flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))

    alpha_mask = (rotated[:, :, 3] / 255.0) * MASK_ALPHA
    
    # [속도 최적화] 루프 대신 넘파이 연산으로 한 번에 처리
    for c in range(3):
        background[:, :, c] = (alpha_mask * rotated[:, :, c] + (1.0 - alpha_mask) * background[:, :, c]).astype(np.uint8)
    
    return background

# 입 합성 함수 (생략되지 않게 포함)
def overlay_image(background, overlay, cx, cy, scale):
    if overlay is None: return background
    h, w, _ = background.shape
    bw = int(w * scale)
    bh = int(bw * (overlay.shape[0] / overlay.shape[1]))
    img_resized = cv2.resize(overlay, (bw, bh))
    y1, y2 = cy - bh//2, cy + bh//2
    x1, x2 = cx - bw//2, cx + bw//2
    if x1 < 0 or y1 < 0 or x2 > w or y2 > h: return background
    alpha_s = img_resized[:, :, 3] / 255.0
    inv_alpha = 1.0 - alpha_s
    for c in range(3):
        background[y1:y2, x1:x2, c] = (alpha_s * img_resized[:, :, c] + inv_alpha * background[y1:y2, x1:x2, c])
    return background

# --- 메인 루프 (최적화 설정 추가) ---
with mp_face_mesh.FaceMesh(
    static_image_mode=False,        # 동영상 모드 활성화 (속도 핵심)
    max_num_faces=1,
    refine_landmarks=False,          # 정말 정밀한 입술 떨림이 필요없다면 False로 (속도 향상)
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as face_mesh:

    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, _ = image.shape
        # [최적화] 연산량을 줄이기 위해 이미지를 RGB로 변환할 때 크기를 살짝 줄여서 process하고 
        # 나중에 좌표만 원복하는 방법도 있지만, 일단은 그냥 진행합니다.
        results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            
            p0 = get_smoothed_coords('eye_0', landmarks[33].x * w, landmarks[33].y * h)
            p1 = get_smoothed_coords('eye_1', landmarks[263].x * w, landmarks[263].y * h)

            # 최적화된 합성 함수 호출
            image = overlay_mask_fast(image, pre_processed_mask, p0, p1)

            mx_raw = landmarks[13].x * w + offset_x
            my_raw = landmarks[13].y * h + offset_y
            mx, my = get_smoothed_coords('mouth', mx_raw, my_raw)
            image = overlay_image(image, bubble_img, int(mx), int(my), mouth_scale)

        cv2.imshow('Optimized Face Mesh', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [87]:
import cv2
import mediapipe as mp
import numpy as np
import math

# 1. 이미지 로드 및 전처리 (이전과 동일)
bubble_img = cv2.imread('bubble.png', cv2.IMREAD_UNCHANGED)
raw_mask = cv2.imread('mask.png', cv2.IMREAD_UNCHANGED)

if raw_mask is not None and raw_mask.shape[2] == 4:
    kernel = np.ones((2,2), np.uint8)
    channels = list(cv2.split(raw_mask))
    channels[3] = cv2.erode(channels[3], kernel, iterations=1)
    pre_processed_mask = cv2.merge(channels)
    pre_processed_mask = cv2.GaussianBlur(pre_processed_mask, (3, 3), 0)
else:
    pre_processed_mask = raw_mask

mp_face_mesh = mp.solutions.face_mesh
cap = cv2.VideoCapture('treeman_talk.mp4')

# --- [영상 저장 설정 추가] ---
# 원본 영상의 정보 가져오기
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or fps > 60: fps = 30.0  # FPS를 가져오지 못할 경우 기본값 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 코덱 설정 (Windows는 'DIVX', Mac/Linux는 'mp4v' 추천)
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
# 저장할 파일명, 코덱, FPS, 크기 순서
out = cv2.VideoWriter('output_result.mp4', fourcc, fps, (width, height))
# ---------------------------

# [설정 변수/함수 생략 - 이전과 동일]
FIXED_MASK_WIDTH = 280
mask_offset_y, mask_offset_x = -55, 0
mouth_scale, offset_y, offset_x = 0.2, -100, 230
SMOOTHING, MASK_ALPHA = 0.5, 0.9
prev_coords = {'eye_0': None, 'eye_1': None, 'mouth': None}

def get_smoothed_coords(key, curr_x, curr_y):
    if prev_coords[key] is None:
        prev_coords[key] = (curr_x, curr_y)
        return curr_x, curr_y
    px, py = prev_coords[key]
    sx = px * SMOOTHING + curr_x * (1 - SMOOTHING)
    sy = py * SMOOTHING + curr_y * (1 - SMOOTHING)
    prev_coords[key] = (sx, sy)
    return sx, sy

def overlay_mask_fast(background, mask, p0, p1):
    if mask is None: return background
    bg_h, bg_w = background.shape[:2]
    cx = (p0[0] + p1[0]) / 2.0 + mask_offset_x
    cy = (p0[1] + p1[1]) / 2.0 + mask_offset_y
    angle = math.degrees(math.atan2(p1[1] - p0[1], p1[0] - p0[0]))
    h_m, w_m = mask.shape[:2]
    mask_width = FIXED_MASK_WIDTH
    mask_height = int(mask_width * (h_m / w_m))
    resized = cv2.resize(mask, (mask_width, mask_height), interpolation=cv2.INTER_LINEAR)
    M = cv2.getRotationMatrix2D((mask_width / 2, mask_height / 2), -angle, 1.0)
    M[0, 2] += (cx - mask_width / 2)
    M[1, 2] += (cy - mask_height / 2)
    rotated = cv2.warpAffine(resized, M, (bg_w, bg_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))
    alpha_mask = (rotated[:, :, 3] / 255.0) * MASK_ALPHA
    for c in range(3):
        background[:, :, c] = (alpha_mask * rotated[:, :, c] + (1.0 - alpha_mask) * background[:, :, c]).astype(np.uint8)
    return background

def overlay_image(background, overlay, cx, cy, scale):
    if overlay is None: return background
    h, w, _ = background.shape
    bw = int(w * scale)
    bh = int(bw * (overlay.shape[0] / overlay.shape[1]))
    img_resized = cv2.resize(overlay, (bw, bh))
    y1, y2 = cy - bh//2, cy + bh//2
    x1, x2 = cx - bw//2, cx + bw//2
    if x1 < 0 or y1 < 0 or x2 > w or y2 > h: return background
    alpha_s = img_resized[:, :, 3] / 255.0
    inv_alpha = 1.0 - alpha_s
    for c in range(3):
        background[y1:y2, x1:x2, c] = (alpha_s * img_resized[:, :, c] + inv_alpha * background[y1:y2, x1:x2, c])
    return background

# --- 메인 루프 ---
with mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=False) as face_mesh:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        h, w, _ = image.shape
        results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            p0 = get_smoothed_coords('eye_0', landmarks[33].x * w, landmarks[33].y * h)
            p1 = get_smoothed_coords('eye_1', landmarks[263].x * w, landmarks[263].y * h)
            image = overlay_mask_fast(image, pre_processed_mask, p0, p1)

            mx_raw = landmarks[13].x * w + offset_x
            my_raw = landmarks[13].y * h + offset_y
            mx, my = get_smoothed_coords('mouth', mx_raw, my_raw)
            image = overlay_image(image, bubble_img, int(mx), int(my), mouth_scale)

        # [중요] 합성된 현재 프레임을 영상 파일에 씁니다.
        out.write(image)

        cv2.imshow('Recording...', image)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

# 모든 자원 해제 (해제하지 않으면 영상 파일이 깨질 수 있음)
cap.release()
out.release() # 파일 닫기 필수!
cv2.destroyAllWindows()
# print("영상 저장이 완료되었습니다: output_result.mp4")